# Day 31 Task: Train an MLP on MNIST

**Objective**: Train an MLP on MNIST with a train/val split. Handle edge cases efficiently.

## 1. Setup and Data Loading

We download the MNIST dataset, apply normalization transforms, and split it into training (50,000 images), validation (10,000 images), and test sets. We then wrap these in DataLoaders for efficient batch processing.

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split, Subset
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# MNIST dataset
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,)) # Standard MNIST normalization
])

# Load datasets
full_train_dataset = torchvision.datasets.MNIST(root='./data', train=True, transform=transform, download=True)
test_dataset = torchvision.datasets.MNIST(root='./data', train=False, transform=transform, download=True)

# Train/Val split (50k train, 10k val)
train_size = 50000
val_size = 10000
train_dataset, val_dataset = random_split(full_train_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(42))

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")


Using device: cpu


100%|██████████| 9.91M/9.91M [00:05<00:00, 1.92MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 146kB/s]
100%|██████████| 1.65M/1.65M [00:01<00:00, 1.23MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 4.14MB/s]

Train size: 50000, Val size: 10000, Test size: 10000


## 2. Defining the MLP

**Edge Case 1 (Shape Mismatch Resilience):** We will handle the shape inside `forward` using `x.view(x.size(0), -1)` so the model automatically flattens `[B, 1, 28, 28]` inputs to `[B, 784]` without breaking.

In [2]:
class MLP(nn.Module):
    def __init__(self, input_size=784, hidden_size=128, num_classes=10):
        super(MLP, self).__init__()
        self.fc1 = nn.Linear(input_size, hidden_size)
        self.relu = nn.ReLU()
        self.fc2 = nn.Linear(hidden_size, hidden_size // 2)
        self.fc3 = nn.Linear(hidden_size // 2, num_classes)
        
    def forward(self, x):
        # EDGE CASE 1 HANDLING: Flatten automatically if passed a 2D/3D image
        if x.dim() > 2:
            x = x.view(x.size(0), -1)
            
        x = self.fc1(x)
        x = self.relu(x)
        x = self.fc2(x)
        x = self.relu(x)
        x = self.fc3(x)
        return x

model = MLP().to(device)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Testing the edge case handling:
dummy_img = torch.randn(16, 1, 28, 28).to(device) # Shape that would normally break a linear layer
try:
    out = model(dummy_img)
    print("Success! Model handled 4D input cleanly. Output shape:", out.shape)
except Exception as e:
    print("Failed!", e)


Success! Model handled 4D input cleanly. Output shape: torch.Size([16, 10])


## 3. Edge Case 2: Overfitting a Single Batch

Before running the full 50,000 image training loop, it is best practice to overfit a single batch. If the network can't learn one batch perfectly (loss approaching ~0.0), there is a bug in the architecture, gradients, or labels.

In [3]:
print("--- Debug: Overfitting a single batch ---")
# Get a single batch
single_batch_inputs, single_batch_targets = next(iter(train_loader))
single_batch_inputs, single_batch_targets = single_batch_inputs.to(device), single_batch_targets.to(device)

debug_model = MLP().to(device)
debug_opt = optim.Adam(debug_model.parameters(), lr=0.005)

for i in range(50):
    debug_opt.zero_grad()
    preds = debug_model(single_batch_inputs)
    loss = criterion(preds, single_batch_targets)
    loss.backward()
    debug_opt.step()
    
    if (i+1) % 10 == 0:
        print(f"Debug Epoch {i+1}, Loss: {loss.item():.6f}")
        
print("Debug check passed: Loss successfully driven near zero.")


--- Debug: Overfitting a single batch ---
Debug Epoch 10, Loss: 0.044641
Debug Epoch 20, Loss: 0.000652
Debug Epoch 30, Loss: 0.000088
Debug Epoch 40, Loss: 0.000029
Debug Epoch 50, Loss: 0.000012
Debug check passed: Loss successfully driven near zero.


## 4. Full Training and Evaluation Loop

This is the core training engine. We iterate through epochs, process batches of images, calculate the loss, backpropagate to get gradients, and update the model weights. After each epoch, we evaluate on the validation set to monitor for overfitting.

In [4]:
epochs = 3

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for inputs, labels in train_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item()
        
    # Validation step
    model.eval()
    val_loss = 0.0
    correct = 0
    total = 0
    with torch.no_grad():
        for inputs, labels in val_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            val_loss += loss.item()
            
            _, predicted = torch.max(outputs.data, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
            
    val_acc = 100 * correct / total
    train_loss = running_loss / len(train_loader)
    val_loss = val_loss / len(val_loader)
    
    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {train_loss:.4f} | Val Loss: {val_loss:.4f} | Val Accuracy: {val_acc:.2f}%")


Epoch [1/3] | Train Loss: 0.2996 | Val Loss: 0.1682 | Val Accuracy: 94.86%
Epoch [2/3] | Train Loss: 0.1233 | Val Loss: 0.1278 | Val Accuracy: 96.16%
Epoch [3/3] | Train Loss: 0.0849 | Val Loss: 0.1023 | Val Accuracy: 96.93%


## 5. Final Test Evaluation

After training is complete, we evaluate the model one last time on the hold-out test set to get an unbiased estimate of its true accuracy.

In [5]:
model.eval()
test_correct = 0
test_total = 0
with torch.no_grad():
    for inputs, labels in test_loader:
        inputs, labels = inputs.to(device), labels.to(device)
        outputs = model(inputs)
        
        _, predicted = torch.max(outputs.data, 1)
        test_total += labels.size(0)
        test_correct += (predicted == labels).sum().item()
        
test_acc = 100 * test_correct / test_total
print(f"Final Test Accuracy: {test_acc:.2f}%")


Final Test Accuracy: 97.17%


## Reflection
*   **What went well:** Using `nn.Sequential`/`Linear` for the MLP is straightforward. Separating the train and validation sets gives a realistic view of model performance.
*   **What was difficult:** Remembering to manage device placement (`.to(device)`) for both the model and the tensors in every loop.
*   **Edge Cases Addressed:** 
    1.  Flattening the 2D input automatically in the `forward` pass prevents common tensor shape mismatches.
    2.  Overfitting a single batch first is an amazing debugging trick to ensure the gradients are flowing properly before waiting for a 50k image epoch to finish.
*   **What remains:** Next step would be adding a CNN instead of an MLP for better image features.